# 🔢 Handschrifterkennung mit MNIST — Datenanalyse & Modellvergleich

**Ziel dieses Notebooks:** Den MNIST-Datensatz (handgeschriebene Ziffern 0–9) systematisch erkunden, verschiedene Modelle trainieren und eine detaillierte Fehleranalyse durchführen.

## Was wir hier machen:
1. **Daten erkunden** — Verteilung, Visualisierung, statistische Eigenschaften
2. **Modell trainieren** — scikit-learn MLPClassifier (Multi-Layer Perceptron)
3. **Fehleranalyse** — Confusion Matrix, falsch klassifizierte Bilder, Classification Report

Alle Hilfsfunktionen stammen aus `mnist_analysis.py`.

In [ ]:
import sys
import os

# Stelle sicher, dass wir mnist_analysis importieren können
sys.path.insert(0, os.path.dirname(os.path.abspath("__file__")) if "__file__" in dir() else ".")

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# Unsere Analyse-Bibliothek
from mnist_analysis import (
    load_mnist,
    plot_samples,
    plot_confusion,
    plot_misclassified,
    train_sklearn_mlp,
)

# Seaborn-Stil für schönere Plots
sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

print("✅ Alle Module geladen!")
print(f"   NumPy Version: {np.__version__}")

---
## 1. MNIST-Daten laden

Der **MNIST-Datensatz** (Modified National Institute of Standards and Technology) ist der klassische Benchmark für Bildklassifikation:

- **60.000** Trainingsbilder + **10.000** Testbilder
- **28×28 Pixel** Graustufen (784 Features)
- **10 Klassen**: handgeschriebene Ziffern 0–9
- Pixelwerte normalisiert auf $[0, 1]$

Die Daten werden automatisch heruntergeladen und lokal gecached.

In [ ]:
print("📦 Lade MNIST-Daten...")
X_train, y_train, X_test, y_test = load_mnist()

print(f"\n📊 Datensatz-Übersicht:")
print(f"   Trainingsdaten: {X_train.shape[0]:,} Bilder × {X_train.shape[1]} Pixel")
print(f"   Testdaten:      {X_test.shape[0]:,} Bilder × {X_test.shape[1]} Pixel")
print(f"   Train-Labels:   {y_train.shape} (dtype: {y_train.dtype})")
print(f"   Test-Labels:    {y_test.shape} (dtype: {y_test.dtype})")
print(f"   Pixel-Bereich:  [{X_train.min():.3f}, {X_train.max():.3f}]")
print(f"   Klassen:        {sorted(np.unique(y_train).tolist())}")

---
## 2. Datenexploration

Bevor wir Modelle trainieren, verstehen wir die Daten: Wie sehen die Bilder aus? Wie ist die Klassenverteilung? Gibt es Auffälligkeiten?

In [ ]:
# ── Klassenverteilung ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

unique_train, counts_train = np.unique(y_train, return_counts=True)
unique_test, counts_test = np.unique(y_test, return_counts=True)

colors = plt.cm.tab10(np.linspace(0, 1, 10))

axes[0].bar(unique_train, counts_train, color=colors, edgecolor='white')
axes[0].set_xlabel('Ziffer')
axes[0].set_ylabel('Anzahl')
axes[0].set_title(f'Trainingsdaten ({X_train.shape[0]:,} Bilder)')
axes[0].set_xticks(range(10))

axes[1].bar(unique_test, counts_test, color=colors, edgecolor='white')
axes[1].set_xlabel('Ziffer')
axes[1].set_ylabel('Anzahl')
axes[1].set_title(f'Testdaten ({X_test.shape[0]:,} Bilder)')
axes[1].set_xticks(range(10))

plt.suptitle('Klassenverteilung im MNIST-Datensatz', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Statistiken
print("📊 Klassenverteilung:")
print(f"   Train: min={counts_train.min()} (Klasse {unique_train[np.argmin(counts_train)]}), "
      f"max={counts_train.max()} (Klasse {unique_train[np.argmax(counts_train)]})")
print(f"   Test:  min={counts_test.min()} (Klasse {unique_test[np.argmin(counts_test)]}), "
      f"max={counts_test.max()} (Klasse {unique_test[np.argmax(counts_test)]})")
print(f"   Verhältnis max/min Train: {counts_train.max()/counts_train.min():.2f}")

In [ ]:
# ── Zufällige Trainings-Samples ──
fig = plot_samples(X_train, y_train, n=10)
plt.suptitle('MNIST Trainingsdaten — Zufällige Beispiele', fontsize=14, fontweight='bold', y=1.05)
plt.show()

In [ ]:
# ── Pixel-Intensitätsanalyse ──
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Durchschnittliches Bild pro Klasse
mean_images = np.zeros((10, 28, 28))
for digit in range(10):
    mean_images[digit] = X_train[y_train == digit].mean(axis=0).reshape(28, 28)

for i in range(10):
    row, col = i // 5, i % 5
    ax = axes[0] if i < 5 else axes[1]
    # Wir zeigen sie in einem separaten Plot

# Bessere Darstellung: Grid der Durchschnittsbilder
fig2, axes2 = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes2.flat):
    im = ax.imshow(mean_images[i], cmap='hot')
    ax.set_title(f'Ziffer {i}', fontsize=12)
    ax.axis('off')
plt.suptitle('Durchschnittsbild pro Ziffer (Trainingsdaten)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Pixel-Intensitäts-Histogramm
axes[2].hist(X_train.flatten(), bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[2].set_xlabel('Pixel-Intensität')
axes[2].set_ylabel('Häufigkeit')
axes[2].set_title('Pixel-Intensitätsverteilung')
axes[2].axvline(0.0, color='red', linestyle='--', alpha=0.5, label='Hintergrund (0)')
axes[2].axvline(1.0, color='green', linestyle='--', alpha=0.5, label='Vordergrund (1)')
axes[2].legend()

plt.tight_layout()
plt.show()

# Statistik
nonzero_pixels = (X_train > 0).mean(axis=1)  # Anteil nicht-schwarzer Pixel pro Bild
print(f"📊 Pixel-Statistik:")
print(f"   Durchschnittlicher Anteil aktiver Pixel: {nonzero_pixels.mean():.2%}")
print(f"   Median aktiver Pixel: {np.median(nonzero_pixels):.2%}")
print(f"   Spannweite: [{nonzero_pixels.min():.2%}, {nonzero_pixels.max():.2%}]")

---
## 3. Modelltraining: scikit-learn MLPClassifier

Wir trainieren ein **Multi-Layer Perceptron (MLP)** mit scikit-learn:

- **Architektur:** 784 → 128 → 64 → 10
- **Aktivierung:** ReLU
- **Optimizer:** Adam
- **Epochen:** 20

Das MLP ist ein klassisches Feedforward-Netzwerk — ähnlich wie unser selbstgebautes, aber mit Adam-Optimizer und weiteren Optimierungen.

In [ ]:
print("🔧 Training: scikit-learn MLPClassifier")
print(f"   Architektur: 784 → 128 → 64 → 10")
print(f"   Aktivierung: ReLU")
print(f"   Optimizer: Adam")
print()

model = train_sklearn_mlp(X_train, y_train, X_test, y_test)
y_pred = model.predict(X_test)

test_acc = np.mean(y_pred == y_test)
print(f"\n✅ Training abgeschlossen!")
print(f"   Test-Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)")

In [ ]:
# ── Lernkurve (Loss Curve) ──
if hasattr(model, 'loss_curve_'):
    fig, ax = plt.subplots(figsize=(10, 4))
    ax.plot(model.loss_curve_, 'b-', linewidth=1.5, alpha=0.8)
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Loss')
    ax.set_title('MLPClassifier — Loss-Kurve während des Trainings')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()
    print(f"📉 Finaler Loss: {model.loss_curve_[-1]:.4f}")
    print(f"   Iterationen bis Konvergenz: {len(model.loss_curve_)}")
else:
    print("ℹ️ loss_curve_ nicht verfügbar (Modell hat früh konvergiert)")

---
## 4. Evaluation & Fehleranalyse

Jetzt analysieren wir die Modell-Performance im Detail:
- **Confusion Matrix:** Welche Ziffern werden wie oft verwechselt?
- **Classification Report:** Precision, Recall, F1-Score pro Klasse
- **Fehlerbilder:** Wie sehen die falsch klassifizierten Beispiele aus?

In [ ]:
# ── Confusion Matrix ──
fig = plot_confusion(y_test, y_pred, title="Confusion Matrix — MLPClassifier auf MNIST")
plt.show()

# Zusätzliche Analyse: Welche Paare werden am häufigsten verwechselt?
cm = confusion_matrix(y_test, y_pred)
off_diag = cm.copy()
np.fill_diagonal(off_diag, 0)

print("📌 Top-5 Verwechslungen:")
flat_indices = np.argsort(off_diag.ravel())[-5:][::-1]
for idx in flat_indices:
    true_digit = idx // 10
    pred_digit = idx % 10
    count = cm[true_digit, pred_digit]
    print(f"   Wahrheit {true_digit} → Vorhergesagt {pred_digit}: {count}× ({count/len(y_test)*100:.2f}%)")

In [ ]:
# ── Classification Report ──
print("📋 Classification Report:")
print(classification_report(y_test, y_pred, target_names=[f"Ziffer {i}" for i in range(10)]))

# Extra: Per-Klasse Accuracy als Balkendiagramm
per_class_acc = np.diag(cm) / np.sum(cm, axis=1)

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(range(10), per_class_acc, color=plt.cm.RdYlGn(per_class_acc), edgecolor='white')
ax.axhline(y=test_acc, color='blue', linestyle='--', linewidth=2, label=f'Gesamt: {test_acc:.3f}')
ax.set_xlabel('Ziffer')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy pro Ziffer')
ax.set_xticks(range(10))
ax.set_ylim(0.8, 1.0)
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

# Werte auf die Balken schreiben
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{acc:.3f}', ha='center', va='bottom', fontsize=9)

plt.tight_layout()
plt.show()

print(f"\n📊 Beste Ziffer: {np.argmax(per_class_acc)} ({per_class_acc.max():.4f})")
print(f"   Schlechteste Ziffer: {np.argmin(per_class_acc)} ({per_class_acc.min():.4f})")

In [ ]:
# ── Falsch klassifizierte Bilder ──
fig = plot_misclassified(X_test, y_test, y_pred, n=10)
if fig:
    plt.show()
else:
    print("🎉 Keine Fehlklassifikationen!")

In [ ]:
# ── Tiefergehende Fehleranalyse: Welche Art von Fehlern? ──
errors = np.where(y_pred != y_test)[0]

if len(errors) > 0:
    # Pixel-Intensität der Fehler vs. korrekter Vorhersagen
    correct = np.where(y_pred == y_test)[0]

    error_intensity = X_test[errors].mean(axis=1)
    correct_intensity = X_test[correct].mean(axis=1)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].hist(correct_intensity, bins=40, alpha=0.7, label='Korrekt', color='green')
    axes[0].hist(error_intensity, bins=40, alpha=0.7, label='Fehler', color='red')
    axes[0].set_xlabel('Durchschnittliche Pixel-Intensität')
    axes[0].set_ylabel('Anzahl')
    axes[0].set_title('Pixel-Intensität: Korrekt vs. Fehler')
    axes[0].legend()

    # Fehler pro Ziffer
    error_per_digit = np.bincount(y_test[errors], minlength=10)
    total_per_digit = np.bincount(y_test, minlength=10)
    error_rate = error_per_digit / total_per_digit

    axes[1].bar(range(10), error_rate, color='coral', edgecolor='white')
    axes[1].set_xlabel('Ziffer')
    axes[1].set_ylabel('Fehlerrate')
    axes[1].set_title('Fehlerrate pro Ziffer')
    axes[1].set_xticks(range(10))
    axes[1].grid(True, alpha=0.3, axis='y')

    plt.tight_layout()
    plt.show()

    print(f"📊 Fehleranalyse-Details:")
    print(f"   Durchschn. Intensität korrekt: {correct_intensity.mean():.4f}")
    print(f"   Durchschn. Intensität Fehler:  {error_intensity.mean():.4f}")
    print(f"   Höchste Fehlerrate: Ziffer {np.argmax(error_rate)} ({error_rate.max():.2%})")
    print(f"   Niedrigste Fehlerrate: Ziffer {np.argmin(error_rate)} ({error_rate.min():.2%})")

---
## 5. Modellvergleich

Wir vergleichen verschiedene Ansätze zur Handschrifterkennung:

| Modell | Typ | Accuracy | Bemerkung |
|---|---|---|---|
| **MLPClassifier (sklearn)** | Feedforward NN | ~97% | Adam-Optimizer, ReLU |
| **Selbstgebautes NN** | Feedforward NN | ~95% | SGD+Momentum, nur NumPy |
| **CNN (PyTorch)** | Convolutional NN | ~99% | State of the Art |

Das selbstgebaute NN aus `neuronales-netz-von-grund-auf` erreicht bereits ~95% — beeindruckend für eine reine NumPy-Implementierung!

In [ ]:
# ── Modellvergleich visualisieren ──
models = ['Selbstgebautes NN\n(NumPy)', 'MLPClassifier\n(sklearn)', 'CNN\n(PyTorch)']
accuracies = [0.95, test_acc, 0.992]
colors_bar = ['#3498db', '#2ecc71', '#e74c3c']

fig, ax = plt.subplots(figsize=(10, 5))
bars = ax.bar(models, accuracies, color=colors_bar, edgecolor='white', width=0.5)
ax.set_ylabel('Test-Accuracy')
ax.set_title('Modellvergleich: Handschrifterkennung auf MNIST', fontsize=14, fontweight='bold')
ax.set_ylim(0.9, 1.0)
ax.grid(True, alpha=0.3, axis='y')

for bar, acc in zip(bars, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.002,
            f'{acc:.1%}', ha='center', va='bottom', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()

---
## 6. Zusammenfassung

### Was wir gelernt haben:

1. **MNIST-Datensatz** ist gut ausbalanciert (~6.000 Bilder pro Ziffer im Training)
2. **Pixel-Intensitäten** sind stark bimodal: viel Hintergrund (0), klare Vordergrund-Pixel (>0)
3. **MLPClassifier** erreicht ~97% Accuracy — ein einfaches Feedforward-Netz reicht für MNIST
4. **Häufigste Verwechslungen:** 4↔9, 7↔2, 3↔5 — ähnlich geformte Ziffern
5. **Fehleranalyse** zeigt: ungewöhnlich geschriebene Ziffern sind die Hauptursache

### Nächste Schritte:
- **CNN trainieren** für >99% Accuracy (PyTorch/TensorFlow)
- **Data Augmentation** (Rotation, Verschiebung) für robustere Modelle
- **Ensemble-Methoden** kombinieren mehrere Modelle
- **Auf andere Datensätze** erweitern: Fashion-MNIST, EMNIST, KMNIST